Explanatory text

## Initialization

In [ ]:
# Imports

from math import exp
from itertools import starmap
from pathlib import Path
from typing import Callable, TypeVar, Any
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import deconvolve

from data_processing.arc_paths import (
    get_parq_root, get_exp_root, INPUT_DATA_FOLDER
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.dataframe_validation import DetectorDataframeColumn
from data_processing.experiment_data_keys import ExperimentDataKey
from data_processing.helpers import stop, get_input_with_default
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.figure_of_merit import gaussian
from data_processing.processing.neutron_classification import classify
from data_processing.processing.neutron_window_generation import (
    generate_nasa_neutron_window,
    generate_n_distro_neutron_window
)
from data_processing.reporting.plotting import plot_classification
from data_processing.helpers import stop, get_input_with_default, input_experiment_ids

## Experiment ID Input

In [ ]:
experiment_ids = input_experiment_ids()
# done = False
# experiment_ids = []
# while not done:
#     while (
#         id_input := input(
#             "Enter the ID number (just the number!), or Enter to finish: "
#         )
#     ) != '':
#         experiment_ids.append(id_input)
#     experiment_ids = [f"ID-{exp_id}" for exp_id in experiment_ids]

#     ids_valid = []
#     for exp_id in experiment_ids:
#         id_valid = get_parq_root(exp_id).is_dir()
#         ids_valid.append(id_valid)
#         if not id_valid:
#             print(f"Experiment {exp_id} cannot be found")

#     done = all(ids_valid)
#     if not done:
#         print("Invalid experiment IDs, please re-enter")
#         experiment_ids = []
#     else:
#         print("All experiment IDs are valid")

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

## Data Loading and Initial Processing

### Neutron Data Processing